In [1]:
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

# Library imports
import torch
import time
import numpy

In [2]:
# Import necessary utilities from my_small_PFN
from scripts.my_small_PFN import MyRegressorPFN

# Create a model with the trained weights
model = MyRegressorPFN("v2.0").set_device("cpu")

Model config' correctly loaded from C:\Users\PC\Desktop\My-Small-PFN\weights\v2.0\my_PFN_config.json
Model weights correctly loaded from C:\Users\PC\Desktop\My-Small-PFN\weights\v2.0\my_PFN_weights.pth


In [3]:
# Let's first create a small test procedure we can use for all datasets
# This follows a similar structure to the one proposed by TabPFN but with 
# normalization and an 80/20 train/test splits.

# First prepare the train and test splits
from sklearn.model_selection import train_test_split
def prepare_splits(X, y, seeds: list[int] = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]):
    # Convert to torch
    X = torch.as_tensor(X, dtype=torch.float32)
    y = torch.as_tensor(y, dtype=torch.float32)

    # Train-test split
    X_train = torch.zeros([0, 0, 0], dtype=torch.float32)
    X_test  = torch.zeros([0, 0, 0], dtype=torch.float32)
    y_train = torch.zeros([0, 0, 0], dtype=torch.float32)
    y_test  = torch.zeros([0, 0, 0], dtype=torch.float32)
    # Iterate through the random seeds
    for seed in seeds:
        # Get the current batch splits
        X_train_batch, X_test_batch, y_train_batch, y_test_batch = train_test_split(X, y, test_size=0.2, random_state=seed)
        # Reshape tensors
        X_train = X_train.view([-1, X_train_batch.shape[0], X_train_batch.shape[1]])
        X_test  =  X_test.view([-1,  X_test_batch.shape[0],  X_test_batch.shape[1]])
        y_train = y_train.view([-1, y_train_batch.shape[0]])
        y_test  =  y_test.view([-1,  y_test_batch.shape[0]])
        # Concatenate
        X_train = torch.cat([X_train, X_train_batch.unsqueeze(0)], dim=0)
        X_test  = torch.cat([ X_test,  X_test_batch.unsqueeze(0)], dim=0)
        y_train = torch.cat([y_train, y_train_batch.unsqueeze(0)], dim=0)
        y_test  = torch.cat([ y_test,  y_test_batch.unsqueeze(0)], dim=0)

    # Normalize
    x_mean = X_train.mean(dim=1, keepdim=True)
    x_std  = X_train.std(dim=1,  keepdim=True)
    y_mean = y_train.mean(dim=1, keepdim=True)
    y_std  = y_train.std(dim=1,  keepdim=True)
    
    X_train = (X_train - x_mean) / x_std
    X_test  = (X_test  - x_mean) / x_std
    y_train = (y_train - y_mean) / y_std
    y_test  = (y_test  - y_mean) / y_std

    # Print data breakdown
    print(f"Total Splits:  {len(seeds)} Batches")
    print(f"Total Rows:    {X.shape[0]} ({X_train.shape[1]} Train, {X_test.shape[1]} Test)")
    print(f"Total Columns: {X.shape[1]} Features + 1 Target\n")

    # Return splits
    return X_train, X_test, y_train, y_test

# Then run the splits through the model and evaluate results
from sklearn.metrics import mean_squared_error, r2_score
def train_test_model(X_train: torch.Tensor, X_test: torch.Tensor, y_train: torch.Tensor, y_test: torch.Tensor):

    # Print and start time
    print("Predicting test target...")
    t0 = time.time()

    # Fit data
    model.fit(X_train, y_train)

    # Predict on the test set
    predictions = model.predict(X_test, output='mean')

    # Evaluate the model
    mses = [mean_squared_error(y_test[b], predictions[b]) for b in range(X_train.shape[0])]
    r2s = [r2_score(y_test[b], predictions[b]) for b in range(X_train.shape[0])]

    # Print values
    print(f"Finished predicting in {(time.time()-t0):.2f}s")
    print("\nMy Small PFN:")
    print(f"Mean Squared Error: {numpy.mean(mses):.4f} ± {numpy.std(mses):.4f}")
    print(f"R² Score:           {numpy.mean( r2s):.4f} ± {numpy.std( r2s):.4f}")

# We also add some baselines to compare to, those will be linear regression and HGBR
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
def baseline_comparissons(X_train: torch.Tensor, X_test: torch.Tensor, y_train: torch.Tensor, y_test: torch.Tensor):
    linear_mses = []
    linear_r2s = []

    hgbr_mses = []
    hgbr_r2s = []

    for b in range(X_train.shape[0]):
        Xtr = X_train[b]
        ytr = y_train[b]
        Xte = X_test[b]
        yte = y_test[b]

        linear = LinearRegression()
        linear.fit(Xtr, ytr)
        linear_preds = linear.predict(Xte)

        linear_mses.append(mean_squared_error(yte, linear_preds))
        linear_r2s.append(r2_score(yte, linear_preds))        

        hgbr = HistGradientBoostingRegressor(max_depth=6)
        hgbr.fit(Xtr, ytr)
        hgbr_preds = hgbr.predict(Xte)

        hgbr_mses.append(mean_squared_error(yte, hgbr_preds))
        hgbr_r2s.append(r2_score(yte, hgbr_preds))

    print("\nLinear Regression:")
    print(f"Mean Squared Error: {numpy.mean(linear_mses):.4f} ± {numpy.std(linear_mses):.4f}")
    print(f"R² Score:           {numpy.mean(linear_r2s ):.4f} ± {numpy.std(linear_r2s ):.4f}")

    print("\nHist Gradient Boosting Regressor:")
    print(f"Mean Squared Error: {numpy.mean(hgbr_mses):.4f} ± {numpy.std(hgbr_mses):.4f}")
    print(f"R² Score:           {numpy.mean(hgbr_r2s ):.4f} ± {numpy.std(hgbr_r2s ):.4f}")

In [4]:
from sklearn.datasets import load_diabetes

# Load data
X, y = load_diabetes(return_X_y=True)

# Split data
X_train, X_test, y_train, y_test = prepare_splits(X, y)

# Predict and analyze results
train_test_model(X_train, X_test, y_train, y_test)

# Also predict with some baselines
baseline_comparissons(X_train, X_test, y_train, y_test)

Total Splits:  10 Batches
Total Rows:    442 (353 Train, 89 Test)
Total Columns: 10 Features + 1 Target

Predicting test target...
Finished predicting in 5.40s

My Small PFN:
Mean Squared Error: 0.4544 ± 0.0614
R² Score:           0.5058 ± 0.0641

Linear Regression:
Mean Squared Error: 0.4600 ± 0.0666
R² Score:           0.5006 ± 0.0633

Hist Gradient Boosting Regressor:
Mean Squared Error: 0.5468 ± 0.0721
R² Score:           0.4046 ± 0.0848


In [5]:
from sklearn.datasets import fetch_california_housing

# Load data
X, y = fetch_california_housing(return_X_y=True)

# Split data
X_train, X_test, y_train, y_test = prepare_splits(X, y)

# Predict and analyze results
train_test_model(X_train, X_test, y_train, y_test)

# Also predict with some baselines
baseline_comparissons(X_train, X_test, y_train, y_test)

Total Splits:  10 Batches
Total Rows:    20640 (16512 Train, 4128 Test)
Total Columns: 8 Features + 1 Target

Predicting test target...
Finished predicting in 1646.75s

My Small PFN:
Mean Squared Error: 0.2394 ± 0.0055
R² Score:           0.7608 ± 0.0078

Linear Regression:
Mean Squared Error: 0.3953 ± 0.0096
R² Score:           0.6051 ± 0.0124

Hist Gradient Boosting Regressor:
Mean Squared Error: 0.1691 ± 0.0060
R² Score:           0.8310 ± 0.0079


In [6]:
from sklearn.datasets import fetch_openml

# Load data (longley)
data = fetch_openml(name="longley", version=1, as_frame=False)
X, y = data.data, data.target

# Split data
X_train, X_test, y_train, y_test = prepare_splits(X, y)

# Predict and analyze results
train_test_model(X_train, X_test, y_train, y_test)

# Also predict with some baselines
baseline_comparissons(X_train, X_test, y_train, y_test)

Total Splits:  10 Batches
Total Rows:    16 (12 Train, 4 Test)
Total Columns: 6 Features + 1 Target

Predicting test target...
Finished predicting in 0.18s

My Small PFN:
Mean Squared Error: 0.0189 ± 0.0136
R² Score:           0.9522 ± 0.0577

Linear Regression:
Mean Squared Error: 0.0149 ± 0.0055
R² Score:           0.9691 ± 0.0246

Hist Gradient Boosting Regressor:
Mean Squared Error: 0.9400 ± 0.4872
R² Score:           -0.4640 ± 0.5242


In [7]:
from sklearn.datasets import fetch_openml

# Load data (cpu_small)
data = fetch_openml(name="cpu_small", version=1, as_frame=False)
X, y = data.data, data.target

# Split data
X_train, X_test, y_train, y_test = prepare_splits(X, y)

# Predict and analyze results
train_test_model(X_train, X_test, y_train, y_test)

# Also predict with some baselines
baseline_comparissons(X_train, X_test, y_train, y_test)

Total Splits:  10 Batches
Total Rows:    8192 (6553 Train, 1639 Test)
Total Columns: 12 Features + 1 Target

Predicting test target...
Finished predicting in 342.25s

My Small PFN:
Mean Squared Error: 0.1130 ± 0.0068
R² Score:           0.8827 ± 0.0015

Linear Regression:
Mean Squared Error: 0.2784 ± 0.0188
R² Score:           0.7108 ± 0.0134

Hist Gradient Boosting Regressor:
Mean Squared Error: 0.0212 ± 0.0011
R² Score:           0.9779 ± 0.0021


In [8]:
from sklearn.datasets import fetch_openml

# Load data (Boston Housing)
data = fetch_openml(name="boston", version=1, as_frame=False)
X, y = data.data.astype(float), data.target

# Split data
X_train, X_test, y_train, y_test = prepare_splits(X, y)

# Predict and analyze results
train_test_model(X_train, X_test, y_train, y_test)

# Also predict with some baselines
baseline_comparissons(X_train, X_test, y_train, y_test)

Total Splits:  10 Batches
Total Rows:    506 (404 Train, 102 Test)
Total Columns: 13 Features + 1 Target

Predicting test target...
Finished predicting in 4.75s

My Small PFN:
Mean Squared Error: 0.1507 ± 0.0441
R² Score:           0.8548 ± 0.0357

Linear Regression:
Mean Squared Error: 0.2877 ± 0.0602
R² Score:           0.7244 ± 0.0417

Hist Gradient Boosting Regressor:
Mean Squared Error: 0.1230 ± 0.0285
R² Score:           0.8818 ± 0.0193


In [9]:
from sklearn.datasets import fetch_openml

# Load data (concrete strength)
data = fetch_openml(name="concrete_compressive_strength", version=7, as_frame=False)
X, y = data.data, data.target

# Split data
X_train, X_test, y_train, y_test = prepare_splits(X, y)

# Predict and analyze results
train_test_model(X_train, X_test, y_train, y_test)

# Also predict with some baselines
baseline_comparissons(X_train, X_test, y_train, y_test)

Total Splits:  10 Batches
Total Rows:    1030 (824 Train, 206 Test)
Total Columns: 8 Features + 1 Target

Predicting test target...
Finished predicting in 8.71s

My Small PFN:
Mean Squared Error: 0.1227 ± 0.0119
R² Score:           0.8760 ± 0.0119

Linear Regression:
Mean Squared Error: 0.3879 ± 0.0341
R² Score:           0.6087 ± 0.0267

Hist Gradient Boosting Regressor:
Mean Squared Error: 0.0872 ± 0.0169
R² Score:           0.9119 ± 0.0173


In [10]:
from sklearn.datasets import fetch_openml

# Load data (airfoil self noise)
data = fetch_openml(name="airfoil_self_noise", version=1, as_frame=False)
X, y = data.data, data.target

# Split data
X_train, X_test, y_train, y_test = prepare_splits(X, y)

# Predict and analyze results
train_test_model(X_train, X_test, y_train, y_test)

# Also predict with some baselines
baseline_comparissons(X_train, X_test, y_train, y_test)

Total Splits:  10 Batches
Total Rows:    1503 (1202 Train, 301 Test)
Total Columns: 5 Features + 1 Target

Predicting test target...
Finished predicting in 10.86s

My Small PFN:
Mean Squared Error: 0.1314 ± 0.0241
R² Score:           0.8693 ± 0.0219

Linear Regression:
Mean Squared Error: 0.4983 ± 0.0390
R² Score:           0.5021 ± 0.0394

Hist Gradient Boosting Regressor:
Mean Squared Error: 0.0897 ± 0.0071
R² Score:           0.9105 ± 0.0054


In [11]:
from sklearn.datasets import fetch_openml

# Load data (QSAR fish toxicity)
data = fetch_openml(name="qsar_fish_toxicity", version=7, as_frame=False)
X, y = data.data, data.target

# Split data
X_train, X_test, y_train, y_test = prepare_splits(X, y)

# Predict and analyze results
train_test_model(X_train, X_test, y_train, y_test)

# Also predict with some baselines
baseline_comparissons(X_train, X_test, y_train, y_test)

Total Splits:  10 Batches
Total Rows:    908 (726 Train, 182 Test)
Total Columns: 6 Features + 1 Target

Predicting test target...
Finished predicting in 4.13s

My Small PFN:
Mean Squared Error: 0.3945 ± 0.0662
R² Score:           0.5996 ± 0.0485

Linear Regression:
Mean Squared Error: 0.4343 ± 0.0704
R² Score:           0.5584 ± 0.0559

Hist Gradient Boosting Regressor:
Mean Squared Error: 0.3837 ± 0.0738
R² Score:           0.6122 ± 0.0449


In [12]:
from sklearn.datasets import fetch_openml

# Load data (Naval Propulsion Plant)
data = fetch_openml(name="naval_propulsion_plant", version=7, as_frame=False)
X, y = data.data, data.target

# Split data
X_train, X_test, y_train, y_test = prepare_splits(X, y)

# Predict and analyze results
train_test_model(X_train, X_test, y_train, y_test)

# Also predict with some baselines
baseline_comparissons(X_train, X_test, y_train, y_test)

Total Splits:  10 Batches
Total Rows:    11934 (9547 Train, 2387 Test)
Total Columns: 14 Features + 1 Target

Predicting test target...
Finished predicting in 839.99s

My Small PFN:
Mean Squared Error: 0.2356 ± 0.0115
R² Score:           0.7630 ± 0.0093

Linear Regression:
Mean Squared Error: 0.2005 ± 0.0036
R² Score:           0.7983 ± 0.0033

Hist Gradient Boosting Regressor:
Mean Squared Error: 0.0126 ± 0.0007
R² Score:           0.9873 ± 0.0005


In [13]:
from sklearn.datasets import fetch_openml

# Load data (Naval Propulsion Plant Limited)
data = fetch_openml(name="naval_propulsion_plant", version=7, as_frame=False)
X, y = data.data[:512+128], data.target[:512+128]

# Split data
X_train, X_test, y_train, y_test = prepare_splits(X, y)

# Predict and analyze results
train_test_model(X_train, X_test, y_train, y_test)

# Also predict with some baselines
baseline_comparissons(X_train, X_test, y_train, y_test)

Total Splits:  10 Batches
Total Rows:    640 (512 Train, 128 Test)
Total Columns: 14 Features + 1 Target

Predicting test target...
Finished predicting in 6.60s

My Small PFN:
Mean Squared Error: 1.0646 ± 0.0690
R² Score:           -0.0590 ± 0.0346

Linear Regression:
Mean Squared Error: 0.9093 ± 0.0733
R² Score:           0.0933 ± 0.0821

Hist Gradient Boosting Regressor:
Mean Squared Error: 0.9710 ± 0.0566
R² Score:           0.0341 ± 0.0163


In [14]:
from sklearn.datasets import fetch_openml

# Load data (yatch hydrodynamics)
data = fetch_openml(name="yacht_hydrodynamics", version=1, as_frame=False)
X, y = data.data, data.target

# Split data
X_train, X_test, y_train, y_test = prepare_splits(X, y)

# Predict and analyze results
train_test_model(X_train, X_test, y_train, y_test)

# Also predict with some baselines
baseline_comparissons(X_train, X_test, y_train, y_test)

Total Splits:  10 Batches
Total Rows:    308 (246 Train, 62 Test)
Total Columns: 6 Features + 1 Target

Predicting test target...
Finished predicting in 1.40s

My Small PFN:
Mean Squared Error: 0.0054 ± 0.0027
R² Score:           0.9953 ± 0.0016

Linear Regression:
Mean Squared Error: 0.3969 ± 0.0916
R² Score:           0.6302 ± 0.0408

Hist Gradient Boosting Regressor:
Mean Squared Error: 0.0780 ± 0.0371
R² Score:           0.9337 ± 0.0225


In [15]:
from sklearn.datasets import fetch_openml

# Load data (kin8nm)
data = fetch_openml(name="kin8nm", version=1, as_frame=False)
X, y = data.data, data.target

# Split data
X_train, X_test, y_train, y_test = prepare_splits(X, y)

# Predict and analyze results
train_test_model(X_train, X_test, y_train, y_test)

# Also predict with some baselines
baseline_comparissons(X_train, X_test, y_train, y_test)

Total Splits:  10 Batches
Total Rows:    8192 (6553 Train, 1639 Test)
Total Columns: 8 Features + 1 Target

Predicting test target...
Finished predicting in 273.04s

My Small PFN:
Mean Squared Error: 0.2570 ± 0.0120
R² Score:           0.7411 ± 0.0116

Linear Regression:
Mean Squared Error: 0.5872 ± 0.0211
R² Score:           0.4088 ± 0.0152

Hist Gradient Boosting Regressor:
Mean Squared Error: 0.2602 ± 0.0131
R² Score:           0.7380 ± 0.0126


In [16]:
from sklearn.datasets import fetch_openml

# Load data (wine quality red)
data = fetch_openml(name="wine-quality-red", version=1, as_frame=False)
X, y = data.data, data.target.astype(float)

# Split data
X_train, X_test, y_train, y_test = prepare_splits(X, y)

# Predict and analyze results
train_test_model(X_train, X_test, y_train, y_test)

# Also predict with some baselines
baseline_comparissons(X_train, X_test, y_train, y_test)

Total Splits:  10 Batches
Total Rows:    1599 (1279 Train, 320 Test)
Total Columns: 11 Features + 1 Target

Predicting test target...
Finished predicting in 19.29s

My Small PFN:
Mean Squared Error: 0.6255 ± 0.0687
R² Score:           0.3653 ± 0.0319

Linear Regression:
Mean Squared Error: 0.6543 ± 0.0755
R² Score:           0.3361 ± 0.0404

Hist Gradient Boosting Regressor:
Mean Squared Error: 0.5695 ± 0.0606
R² Score:           0.4210 ± 0.0433


In [17]:
from sklearn.datasets import fetch_openml

# Load data (wine quality white)
data = fetch_openml(name="wine-quality-white", version=1, as_frame=False)
X, y = data.data, data.target.astype(float)

# Split data
X_train, X_test, y_train, y_test = prepare_splits(X, y)

# Predict and analyze results
train_test_model(X_train, X_test, y_train, y_test)

# Also predict with some baselines
baseline_comparissons(X_train, X_test, y_train, y_test)

Total Splits:  10 Batches
Total Rows:    4898 (3918 Train, 980 Test)
Total Columns: 11 Features + 1 Target

Predicting test target...
Finished predicting in 127.03s

My Small PFN:
Mean Squared Error: 0.6423 ± 0.0262
R² Score:           0.3571 ± 0.0178

Linear Regression:
Mean Squared Error: 0.7196 ± 0.0350
R² Score:           0.2798 ± 0.0246

Hist Gradient Boosting Regressor:
Mean Squared Error: 0.5504 ± 0.0213
R² Score:           0.4488 ± 0.0209


In [18]:
from sklearn.datasets import fetch_openml

# Load data (Energy efficiency)
data = fetch_openml(name="energy_efficiency", version=1, as_frame=False)
X, y = data.data.astype(float), data.target

# Split data
X_train, X_test, y_train, y_test = prepare_splits(X, y)

# Predict and analyze results
train_test_model(X_train, X_test, y_train, y_test)

# Also predict with some baselines
baseline_comparissons(X_train, X_test, y_train, y_test)

Total Splits:  10 Batches
Total Rows:    768 (614 Train, 154 Test)
Total Columns: 8 Features + 1 Target

Predicting test target...
Finished predicting in 5.49s

My Small PFN:
Mean Squared Error: 0.0219 ± 0.0060
R² Score:           0.9780 ± 0.0061

Linear Regression:
Mean Squared Error: 0.0826 ± 0.0119
R² Score:           0.9174 ± 0.0121

Hist Gradient Boosting Regressor:
Mean Squared Error: 0.0028 ± 0.0007
R² Score:           0.9972 ± 0.0008


In [19]:
from sklearn.datasets import fetch_openml

# Load data (Auto MPG)
data = fetch_openml(name="autoMpg", version=3, as_frame=False)
X, y = data.data, data.target

# Split data
X_train, X_test, y_train, y_test = prepare_splits(X, y)

# Predict and analyze results
train_test_model(X_train, X_test, y_train, y_test)

# Also predict with some baselines
baseline_comparissons(X_train, X_test, y_train, y_test)

Total Splits:  10 Batches
Total Rows:    392 (313 Train, 79 Test)
Total Columns: 5 Features + 1 Target

Predicting test target...
Finished predicting in 2.12s

My Small PFN:
Mean Squared Error: 0.1016 ± 0.0214
R² Score:           0.8837 ± 0.0237

Linear Regression:
Mean Squared Error: 0.1737 ± 0.0344
R² Score:           0.8006 ± 0.0409

Hist Gradient Boosting Regressor:
Mean Squared Error: 0.1179 ± 0.0272
R² Score:           0.8646 ± 0.0323


In [20]:
from sklearn.datasets import make_friedman1

# Load data (Friedman 1: 64 Rows)
X, y = make_friedman1(n_samples=64, random_state=0)

# Split data
X_train, X_test, y_train, y_test = prepare_splits(X, y)

# Predict and analyze results
train_test_model(X_train, X_test, y_train, y_test)

# Also predict with some baselines
baseline_comparissons(X_train, X_test, y_train, y_test)

Total Splits:  10 Batches
Total Rows:    64 (51 Train, 13 Test)
Total Columns: 10 Features + 1 Target

Predicting test target...
Finished predicting in 0.54s

My Small PFN:
Mean Squared Error: 0.0545 ± 0.0242
R² Score:           0.9422 ± 0.0321

Linear Regression:
Mean Squared Error: 0.3333 ± 0.0784
R² Score:           0.6524 ± 0.1143

Hist Gradient Boosting Regressor:
Mean Squared Error: 0.3508 ± 0.0798
R² Score:           0.6139 ± 0.1931


In [21]:
from sklearn.datasets import make_friedman1

# Load data (Friedman 1: 128 Rows)
X, y = make_friedman1(n_samples=128, random_state=0)

# Split data
X_train, X_test, y_train, y_test = prepare_splits(X, y)

# Predict and analyze results
train_test_model(X_train, X_test, y_train, y_test)

# Also predict with some baselines
baseline_comparissons(X_train, X_test, y_train, y_test)

Total Splits:  10 Batches
Total Rows:    128 (102 Train, 26 Test)
Total Columns: 10 Features + 1 Target

Predicting test target...
Finished predicting in 0.98s

My Small PFN:
Mean Squared Error: 0.0385 ± 0.0353
R² Score:           0.9625 ± 0.0287

Linear Regression:
Mean Squared Error: 0.2883 ± 0.1097
R² Score:           0.7042 ± 0.0546

Hist Gradient Boosting Regressor:
Mean Squared Error: 0.1970 ± 0.0896
R² Score:           0.7988 ± 0.0565


In [22]:
from sklearn.datasets import make_friedman2

# Load data (Friedman 2: 64 Rows)
X, y = make_friedman2(n_samples=64, random_state=0)

# Split data
X_train, X_test, y_train, y_test = prepare_splits(X, y)

# Predict and analyze results
train_test_model(X_train, X_test, y_train, y_test)

# Also predict with some baselines
baseline_comparissons(X_train, X_test, y_train, y_test)

Total Splits:  10 Batches
Total Rows:    64 (51 Train, 13 Test)
Total Columns: 4 Features + 1 Target

Predicting test target...
Finished predicting in 0.31s

My Small PFN:
Mean Squared Error: 0.0085 ± 0.0160
R² Score:           0.9940 ± 0.0084

Linear Regression:
Mean Squared Error: 0.1670 ± 0.0714
R² Score:           0.8427 ± 0.0371

Hist Gradient Boosting Regressor:
Mean Squared Error: 0.4130 ± 0.2353
R² Score:           0.6109 ± 0.1297


In [23]:
from sklearn.datasets import make_friedman2

# Load data (Friedman 2: 128 Rows)
X, y = make_friedman2(n_samples=128, random_state=0)

# Split data
X_train, X_test, y_train, y_test = prepare_splits(X, y)

# Predict and analyze results
train_test_model(X_train, X_test, y_train, y_test)

# Also predict with some baselines
baseline_comparissons(X_train, X_test, y_train, y_test)

Total Splits:  10 Batches
Total Rows:    128 (102 Train, 26 Test)
Total Columns: 4 Features + 1 Target

Predicting test target...
Finished predicting in 0.76s

My Small PFN:
Mean Squared Error: 0.0022 ± 0.0028
R² Score:           0.9983 ± 0.0015

Linear Regression:
Mean Squared Error: 0.1521 ± 0.0369
R² Score:           0.8526 ± 0.0610

Hist Gradient Boosting Regressor:
Mean Squared Error: 0.0878 ± 0.0473
R² Score:           0.9150 ± 0.0601


In [24]:
from sklearn.datasets import make_friedman3

# Load data (Friedman 3: 64 Rows)
X, y = make_friedman3(n_samples=64, random_state=0)

# Split data
X_train, X_test, y_train, y_test = prepare_splits(X, y)

# Predict and analyze results
train_test_model(X_train, X_test, y_train, y_test)

# Also predict with some baselines
baseline_comparissons(X_train, X_test, y_train, y_test)

Total Splits:  10 Batches
Total Rows:    64 (51 Train, 13 Test)
Total Columns: 4 Features + 1 Target

Predicting test target...
Finished predicting in 0.31s

My Small PFN:
Mean Squared Error: 0.1063 ± 0.0943
R² Score:           0.8763 ± 0.0729

Linear Regression:
Mean Squared Error: 0.5274 ± 0.2908
R² Score:           0.2129 ± 0.8471

Hist Gradient Boosting Regressor:
Mean Squared Error: 0.7573 ± 0.5016
R² Score:           0.0453 ± 0.7507


In [25]:
from sklearn.datasets import make_friedman3

# Load data (Friedman 3: 128 Rows)
X, y = make_friedman3(n_samples=128, random_state=0)

# Split data
X_train, X_test, y_train, y_test = prepare_splits(X, y)

# Predict and analyze results
train_test_model(X_train, X_test, y_train, y_test)

# Also predict with some baselines
baseline_comparissons(X_train, X_test, y_train, y_test)

Total Splits:  10 Batches
Total Rows:    128 (102 Train, 26 Test)
Total Columns: 4 Features + 1 Target

Predicting test target...
Finished predicting in 0.58s

My Small PFN:
Mean Squared Error: 0.0846 ± 0.0787
R² Score:           0.9181 ± 0.0411

Linear Regression:
Mean Squared Error: 0.4870 ± 0.2523
R² Score:           0.3677 ± 0.2272

Hist Gradient Boosting Regressor:
Mean Squared Error: 0.4525 ± 0.2390
R² Score:           0.3560 ± 0.4407


---

Table of tests results. Format mean $\pm$ std:

| Data Set                          | Rows    | Features | PFN MSE         | PFN R²          | LR MSE          | LR R²           | HGBR MSE        | HGBR R²         |
|-----------------------------------|---------|----------|-----------------|-----------------|-----------------|-----------------|-----------------|-----------------|
| Diabetes                          | 442     | 10       | 0.4544 ± 0.0614 | 0.5058 ± 0.0641 | 0.4600 ± 0.0666 | 0.5006 ± 0.0633 | 0.5468 ± 0.0721 | 0.4046 ± 0.0848 |
| California Housing                | 20640   | 8        | 0.2394 ± 0.0055 | 0.7608 ± 0.0078 | 0.3953 ± 0.0096 | 0.6051 ± 0.0124 | 0.1689 ± 0.0061 | 0.8312 ± 0.0082 |
| Longley                           | 16      | 6        | 0.0189 ± 0.0136 | 0.9522 ± 0.0577 | 0.0149 ± 0.0055 | 0.9691 ± 0.0246 | 0.9400 ± 0.4872 |-0.4640 ± 0.5242 |
| CPU Small                         | 8192    | 12       | 0.1130 ± 0.0068 | 0.8827 ± 0.0015 | 0.2784 ± 0.0188 | 0.7108 ± 0.0134 | 0.0212 ± 0.0011 | 0.9779 ± 0.0021 |
| Boston Housing                    | 506     | 13       | 0.1507 ± 0.0441 | 0.8548 ± 0.0357 | 0.2877 ± 0.0602 | 0.7244 ± 0.0417 | 0.1230 ± 0.0285 | 0.8818 ± 0.0193 |
| Concrete Compressive Strength     | 1030    | 8        | 0.1227 ± 0.0119 | 0.8760 ± 0.0119 | 0.3879 ± 0.0341 | 0.6087 ± 0.0267 | 0.0872 ± 0.0169 | 0.9119 ± 0.0173 |
| Airfoil Self-Noise                | 1503    | 5        | 0.1314 ± 0.0241 | 0.8693 ± 0.0219 | 0.4983 ± 0.0390 | 0.5021 ± 0.0394 | 0.0897 ± 0.0071 | 0.9105 ± 0.0054 |
| QSAR Fish Toxicity                | 908     | 6        | 0.3945 ± 0.0662 | 0.5996 ± 0.0485 | 0.4343 ± 0.0704 | 0.5584 ± 0.0559 | 0.3837 ± 0.0738 | 0.6122 ± 0.0449 |
| Naval Propulsion Plant            | 11934   | 14       | 0.2356 ± 0.0115 | 0.7630 ± 0.0093 | 0.2005 ± 0.0036 | 0.7983 ± 0.0033 | 0.0126 ± 0.0007 | 0.9873 ± 0.0005 |
| Naval Propulsion Plant (512 Rows) | 640     | 14       | 1.0646 ± 0.0690 |-0.0590 ± 0.0346 | 0.9093 ± 0.0733 | 0.0933 ± 0.0821 | 0.9710 ± 0.0566 | 0.0341 ± 0.0163 |
| Yacht Hydrodynamics               | 308     | 6        | 0.0054 ± 0.0027 | 0.9953 ± 0.0016 | 0.3969 ± 0.0916 | 0.6302 ± 0.0408 | 0.0780 ± 0.0371 | 0.9337 ± 0.0225 |
| Kin8rm                            | 8192    | 8        | 0.2570 ± 0.0120 | 0.7411 ± 0.0116 | 0.5872 ± 0.0211 | 0.4088 ± 0.0152 | 0.2602 ± 0.0131 | 0.7380 ± 0.0126 |
| Wine Quality Red                  | 1599    | 11       | 0.6255 ± 0.0687 | 0.3653 ± 0.0319 | 0.6543 ± 0.0755 | 0.3361 ± 0.0404 | 0.5695 ± 0.0606 | 0.4210 ± 0.0433 |
| Wine Quality White                | 4898    | 11       | 0.6423 ± 0.0262 | 0.3571 ± 0.0178 | 0.7196 ± 0.0350 | 0.2798 ± 0.0246 | 0.5504 ± 0.0213 | 0.4488 ± 0.0209 |
| Energy Efficiency                 | 768     | 8        | 0.0219 ± 0.0060 | 0.9780 ± 0.0061 | 0.0826 ± 0.0119 | 0.9174 ± 0.0121 | 0.0028 ± 0.0007 | 0.9972 ± 0.0008 |
| Auto MPG                          | 392     | 5        | 0.1016 ± 0.0214 | 0.8837 ± 0.0237 | 0.1737 ± 0.0344 | 0.8006 ± 0.0409 | 0.1179 ± 0.0272 | 0.8646 ± 0.0323 |
| Friedman #1 (64 Rows)             | 64      | 10       | 0.0545 ± 0.0242 | 0.9422 ± 0.0321 | 0.3333 ± 0.0784 | 0.6524 ± 0.1143 | 0.3508 ± 0.0798 | 0.6139 ± 0.1931 |
| Friedman #1 (128 Rows)            | 128     | 10       | 0.0385 ± 0.0353 | 0.9625 ± 0.0287 | 0.2883 ± 0.1097 | 0.7042 ± 0.0546 | 0.1970 ± 0.0896 | 0.7988 ± 0.0565 |
| Friedman #2 (64 Rows)             | 64      | 10       | 0.0085 ± 0.0160 | 0.9940 ± 0.0084 | 0.1670 ± 0.0714 | 0.8427 ± 0.0371 | 0.4130 ± 0.2353 | 0.6109 ± 0.1297 |
| Friedman #2 (128 Rows)            | 128     | 10       | 0.0022 ± 0.0028 | 0.9983 ± 0.0015 | 0.1521 ± 0.0369 | 0.8526 ± 0.0610 | 0.0878 ± 0.0473 | 0.9150 ± 0.0601 |
| Friedman #3 (64 Rows)             | 64      | 10       | 0.1063 ± 0.0943 | 0.8763 ± 0.0729 | 0.5274 ± 0.2908 | 0.2129 ± 0.8471 | 0.7573 ± 0.5016 | 0.0453 ± 0.7507 |
| Friedman #3 (128 Rows)            | 128     | 10       | 0.0846 ± 0.0787 | 0.9181 ± 0.0411 | 0.4870 ± 0.2523 | 0.3677 ± 0.2272 | 0.4525 ± 0.2390 | 0.3560 ± 0.4407 |